In [ ]:
!wget -O ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip .zip


--2025-05-25 05:37:37--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 13.248.244.96, 35.71.179.82, 75.2.60.68, ...
Connecting to bin.equinox.io (bin.equinox.io)|13.248.244.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9462138 (9.0M) [application/octet-stream]
Saving to: ‘ngrok.zip’

ngrok.zip           100%[===================>]   9.02M  4.47MB/s    in 2.0s    

2025-05-25 05:37:40 (4.47 MB/s) - ‘ngrok.zip’ saved [9462138/9462138]

unzip:  cannot find or open .zip, .zip.zip or .zip.ZIP.


In [ ]:
# prompt: 2025-05-04 21:18:11 (50.9 MB/s) - ‘ngrok.zip’ saved [9462138/9462138]
# unzip:  cannot find or open .zip, .zip.zip or .zip.ZIP.

!wget -O ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip ngrok.zip


--2025-05-25 05:37:40--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 13.248.244.96, 35.71.179.82, 75.2.60.68, ...
Connecting to bin.equinox.io (bin.equinox.io)|13.248.244.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9462138 (9.0M) [application/octet-stream]
Saving to: ‘ngrok.zip’

ngrok.zip           100%[===================>]   9.02M  5.55MB/s    in 1.6s    

2025-05-25 05:37:43 (5.55 MB/s) - ‘ngrok.zip’ saved [9462138/9462138]

Archive:  ngrok.zip
  inflating: ngrok                   


In [ ]:
!./ngrok authtoken 2tTShRI76D34LkpdUh3Fw2wNCKs_5Tso1fBm8GpA82jX8Wuha

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import os
os.makedirs("static/images", exist_ok=True)


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving black.png to black.png
Saving brown.png to brown.png
Saving curls.JPG to curls.JPG
Saving flecks.JPG to flecks.JPG
Saving gray.png to gray.png
Saving mosaic.JPG to mosaic.JPG
Saving pink.png to pink.png
Saving powdery.JPG to powdery.JPG
Saving red.png to red.png
Saving spots.JPG to spots.JPG
Saving stripes.JPG to stripes.JPG
Saving velvety.JPG to velvety.JPG
Saving white.png to white.png
Saving yellow.png to yellow.png


In [ ]:
import os
import sqlite3
from flask import Flask, render_template_string, request, redirect, url_for , jsonify
import spacy
from spacy.tokens import Span
import pandas as pd
from collections import defaultdict
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import threading
import cv2
import time


nlp = spacy.load("/content/drive/MyDrive/output/model-best")
app = Flask(__name__)
DATABASE = 'diseases.db'

classification_model = tf.keras.models.load_model('/content/drive/MyDrive/PlantVillage/classification_model.h5')
semantics_model = tf.keras.models.load_model('/content/drive/MyDrive/PlantVillage/segmentation_model.h5')

def add_disease_entry(database, plant_name, disease_name, semantics, colors_list):
    conn = sqlite3.connect(database)
    c = conn.cursor()

    try:
        # Get semantics_id
        c.execute("SELECT id FROM semantics WHERE value = ?", (semantics,))
        row = c.fetchone()
        if row is None:
            raise ValueError(f"Semantic '{semantics}' not found.")
        semantics_id = row[0]

        # Insert disease
        c.execute("""
            INSERT INTO diseases (plant_name, disease_name, semantics_id)
            VALUES (?, ?, ?)
        """, (plant_name, disease_name, semantics_id))
        disease_id = c.lastrowid

        if colors_list != None:
          # Insert colors
          for color in colors_list:
              if color != "None":
                c.execute("SELECT id FROM colors WHERE value = ?", (color,))
                row = c.fetchone()
                if row is None:
                    raise ValueError(f"Color '{color}' not found.")
                color_id = row[0]

                c.execute("""
                    INSERT OR IGNORE INTO disease_colors (disease_id, color_id)
                    VALUES (?, ?)
                """, (disease_id, color_id))

        conn.commit()
        print(f"✅ Disease '{disease_name}' added successfully.")

    except Exception as e:
        print(f"❌ Error: {e}")
        conn.rollback()

    finally:
        conn.close()

def print_all_disease_records(database):
    import sqlite3

    conn = sqlite3.connect(database)
    c = conn.cursor()

    try:
        # Get all diseases with their semantic labels
        c.execute("""
            SELECT d.id, d.plant_name, d.disease_name, s.value
            FROM diseases d
            JOIN semantics s ON d.semantics_id = s.id
        """)
        diseases = c.fetchall()

        for disease in diseases:
            disease_id, plant_name, disease_name, semantics_value = disease

            # Get associated color names
            c.execute("""
                SELECT c.value
                FROM disease_colors dc
                JOIN colors c ON dc.color_id = c.id
                WHERE dc.disease_id = ?
            """, (disease_id,))
            color_rows = c.fetchall()
            colors = [row[0] for row in color_rows]

            print(f"🌿 Plant: {plant_name}")
            print(f"🦠 Disease: {disease_name}")
            print(f"🧠 Semantic: {semantics_value}")
            print(f"🎨 Colors: {', '.join(colors)}")
            print("-" * 40)

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        conn.close()


def get_disease_full_records(database, plant_name, semantics_value):
    conn = sqlite3.connect(database)
    c = conn.cursor()

    query = '''
        SELECT
            d.id AS disease_id,
            d.plant_name,
            d.disease_name,
            s.value AS semantics_value,
            GROUP_CONCAT(c.value, ', ') AS colors
        FROM diseases d
        JOIN semantics s ON d.semantics_id = s.id
        LEFT JOIN disease_colors dc ON d.id = dc.disease_id
        LEFT JOIN colors c ON dc.color_id = c.id
        WHERE d.plant_name = ? AND s.value = ?
        GROUP BY d.id;
    '''
    c.execute(query, (plant_name, semantics_value))
    results = c.fetchall()

    conn.close()
    return results

def filter_records_by_all_colors(records, required_colors):
    if not required_colors:
        return []

    required_set = set(color.lower() for color in required_colors)

    filtered = []
    for record in records:
        # record = (disease_id, plant_name, disease_name, semantics_value, "color1, color2, ...")
        colors = set(map(str.strip, record[4].lower().split(','))) if record[4] else set()
        if required_set.issubset(colors):
            filtered.append(record)

    return filtered

def extract_disease_names(records):
    return [record[2] for record in records]

def filter_records_by_color_match_priority(records, target_colors):
    if not target_colors:
        return []

    target_set = set(color.lower().strip() for color in target_colors)
    exact_matches = []
    superset_matches = []

    for record in records:
        # record = (disease_id, plant_name, disease_name, semantics_value, "color1, color2, ...")
        record_colors = set(color.strip().lower() for color in record[4].split(',')) if record[4] else set()

        if record_colors == target_set:
            exact_matches.append(record)
        elif target_set.issubset(record_colors):
            superset_matches.append(record)

    if exact_matches:
        return exact_matches
    else:
        return superset_matches




def get_best_disease_by_color_priority(plant, semantics, colors_list ):

  records = get_disease_full_records(
  database= DATABASE,
  plant_name = plant,
  semantics_value=semantics
  )


  if(len(records)==1):
    disease_name = extract_disease_names(records)
    return disease_name[0]

  else:
    if(len(colors_list)==0):
      return "No Disease Found"
    else:
      filtered = filter_records_by_color_match_priority(records, colors_list[:2])
      if(len(filtered)==1):
        disease_name = extract_disease_names(filtered)
        return disease_name[0]
      else:
        filtered = filter_records_by_color_match_priority(records, colors_list[:3])
        if(len(filtered)==1):
          disease_name = extract_disease_names(filtered)
          return disease_name[0]
        else:
          filtered = filter_records_by_color_match_priority(records, colors_list[:4])
          if(len(filtered)==1):
             disease_name = extract_disease_names(filtered)
             return disease_name[0]
          else:
             return "No Disease Found"



def get_best_weighted_iou_match(plant, semantics, colors_list):

    records = get_disease_full_records(
    database=DATABASE,
    plant_name = plant,
    semantics_value=semantics
    )
    colors_array = [color.strip() for color in colors_list.split(',')]
    print("Plant name: ", plant)
    print("Semantics: ", semantics)
    print("colors list:", colors_array)

    # Normalize and create weight dict: earlier colors get higher weights
    normalized_colors = [color.strip().lower() for color in colors_array]
    weight_dict = {
        color: max(0.1, 1.0 - 0.2 * i)  # example: 1.0, 0.8, 0.6, 0.4, ...
        for i, color in enumerate(normalized_colors)
    }

    print("weighted dictionary :",weight_dict)
    print("normalized colors :",normalized_colors)
    best_record = None
    best_score = -1

    for record in records:
        record_colors = set(color.strip().lower() for color in record[4].split(',')) if record[4] else set()
        union = set(normalized_colors) | record_colors

        # Weighted intersection
        weighted_intersection = sum(
            weight_dict.get(color, 0) for color in record_colors if color in weight_dict
        )

        score = weighted_intersection / len(union) if union else 0

        if score > best_score:
            best_score = score
            best_record = record

    return best_record

def init_db():
    """Initializes the database, creates tables, and inserts sample data if needed."""
    conn = sqlite3.connect(DATABASE)
    c = conn.cursor()

    # Create tables
    c.execute('''
        CREATE TABLE IF NOT EXISTS colors (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            value VARCHAR(25) UNIQUE
        );
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS semantics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            value VARCHAR(25) UNIQUE
        );
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS diseases (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            plant_name VARCHAR(25),
            disease_name VARCHAR(25),
            semantics_id INTEGER,
            FOREIGN KEY (semantics_id) REFERENCES semantics(id)
        );
    ''')

    c.execute('''
        CREATE TABLE IF NOT EXISTS disease_colors (
            disease_id INTEGER,
            color_id INTEGER,
            PRIMARY KEY (disease_id, color_id),
            FOREIGN KEY (disease_id) REFERENCES diseases(id),
            FOREIGN KEY (color_id) REFERENCES colors(id)
        );
    ''')
    conn.commit()
    # Check if we already have data (using diseases table as reference)
    c.execute("SELECT COUNT(*) FROM diseases;")
    count = c.fetchone()[0]
    if count == 0:
        colors = ['black', 'brown', 'gray', 'yellow', 'red', 'purple', 'white']
        for color in colors:
            c.execute("INSERT OR IGNORE INTO colors (value) VALUES (?);", (color,))

        semantics = ['spots', 'flecks', 'curls', 'stripes', 'mosaic', 'powdery', 'velvety']
        for sem in semantics:
            c.execute("INSERT OR IGNORE INTO semantics (value) VALUES (?);", (sem,))

        conn.commit()

    conn.close()

# Initialize the database when the app starts
init_db()


def extract_entities(text):

    doc = nlp(text)
    entities_by_label = defaultdict(list)
    for ent in doc.ents:
        entities_by_label[ent.label_].append(ent.text)
        print(f"Identified entity, Label: {ent.label_}, Entity: {ent.text}")
    # Extract each category with default "unknown" if not found
    plant_name = entities_by_label.get("PLANT", ["unknown"])[0]
    disease_name = entities_by_label.get("DISEASE", ["unknown"])[0]

    semantics = entities_by_label.get("SEMANTIC", ["unknown"])[0]


    colors_list = entities_by_label.get("COLOR", [])
    colors = ", ".join(colors_list) if colors_list else None

    return {
        "plant_name": plant_name,
        "disease_name": disease_name,
        "semantics": semantics,
        "colors": colors
    }

# Template for the entities confirmation page
entities_template = '''
<!DOCTYPE html>
<html>
<head>
    <title>Identified Entities</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            margin: 0;
            padding: 0;
            background: #f7f7f7;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
        }
        .container {
            width: 50%;
            max-width: 600px;
            background: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 4px 10px rgba(0,0,0,0.1);
            text-align: center;
        }
        form {
            display: flex;
            flex-direction: column;
            align-items: center;
        }
        label {
            width: 100%;
            text-align: left;
            margin-top: 10px;
            font-weight: bold;
        }
        input[type="text"] {
            padding: 8px;
            width: 90%;
            border-radius: 5px;
            border: 1px solid #ccc;
            box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
        }
        input[type="submit"] {
            margin-top: 20px;
            padding: 12px 24px;
            background-color: #4CAF50;
            border: none;
            color: white;
            border-radius: 25px;
            cursor: pointer;
            transition: background 0.3s;
        }
        input[type="submit"]:hover {
            background-color: #45a049;
        }
        .error {
            color: red;
            margin-top: 10px;
        }
        .button-section {
            margin-top: 20px;
        }
        .button-section a {
            text-decoration: none;
            color: #4CAF50;
            font-weight: bold;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>Identified Entities</h1>
        {% if error %}
          <div class="error">{{ error }}</div>
        {% endif %}
        <form method="post" action="{{ url_for('confirm') }}">
            <label for="plant_name">Plant Name:</label>
            <input type="text" name="plant_name" id="plant_name" value="{{ plant_name }}" required>
            <label for="disease_name">Disease Name:</label>
            <input type="text" name="disease_name" id="disease_name" value="{{ disease_name }}" required>
            <label for="semantics">Semantics:</label>
            <input type="text" name="semantics" id="semantics" value="{{ semantics }}" required>
            <label for="colors">Colors (comma separated):</label>
            <input type="text" name="colors" id="colors" value="{{ colors }}" required>
            <div class="button-section">
                <input type="submit" value="Accept">
            </div>
        </form>
        <div class="button-section">
            <a href="{{ url_for('add') }}">Back to Add Page</a>
        </div>
    </div>
</body>
</html>
'''

@app.route('/')
def index():
    """Displays the home page with a table of diseases and buttons for additional actions."""
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    c = conn.cursor()

    query = '''
        SELECT
            d.id,
            d.plant_name,
            d.disease_name,
            s.value AS semantics,
            group_concat(DISTINCT col.value) AS colors
        FROM diseases d
        LEFT JOIN semantics s ON d.semantics_id = s.id
        LEFT JOIN disease_colors dc ON d.id = dc.disease_id
        LEFT JOIN colors col ON dc.color_id = col.id
        GROUP BY d.id, d.plant_name, d.disease_name, s.value;

    '''
    c.execute(query)
    diseases = c.fetchall()
    conn.close()

    html_template = '''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Semantic-Search</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 0;
                background: #f7f7f7;
                display: flex;
                justify-content: center;
                align-items: center;
                min-height: 100vh;
            }
            .container {
                width: 80%;
                max-width: 900px;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 4px 10px rgba(0,0,0,0.1);
                text-align: center;
            }
            h1, h2 {
                color: #333;
            }
            .button-section {
                margin-bottom: 20px;
            }
            .button-section a {
                margin: 0 10px;
                text-decoration: none;
            }
            .button-section button {
                padding: 12px 24px;
                font-size: 16px;
                cursor: pointer;
                background-color: #4CAF50;
                border: none;
                color: white;
                border-radius: 25px;
                transition: background 0.3s;
            }
            .button-section button:hover {
                background-color: #45a049;
            }
            .table-container {
                overflow-x: auto;
                margin-top: 20px;
            }
            .guide-section {
                margin-top: 40px;
                text-align: left;
            }
            .patterns {
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(170px, 1fr));
                gap: 15px;
            }
            .pattern {
                background: #f1f1f1;
                padding: 10px;
                border-radius: 8px;
            }
            .pattern img {
                margin-top: 5px;
                width: 100%;
                border-radius: 5px;
                object-fit: cover;
            }
            .color-spectrum {
                display: flex;
                flex-wrap: wrap;
                gap: 30px;
                margin-top: 20px;
            }
            .color-block {
                flex: 1;
                min-width: 200px;
            }
            .color-block div {
                margin-top: 5px;
            }

            table {
                width: 100%;
                border-collapse: collapse;
                border-radius: 10px;
                overflow: hidden;
                box-shadow: 0 2px 8px rgba(0,0,0,0.1);
                background: #fff;
            }
            th, td {
                padding: 15px;
                text-align: left;
            }
            th {
                background-color: #4CAF50;
                color: white;
            }
            tr:nth-child(even) {
                background-color: #f9f9f9;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Semantic-Search</h1>
            <div class="button-section">
                <a href="/classify"><button>Classify a Leaf</button></a>
                <a href="/add"><button>Add a New Disease</button></a>
            </div>
            <h2>Available Diseases</h2>
            <div class="table-container">
                <table>
                    <tr>
                        <th>ID</th>
                        <th>Plant Name</th>
                        <th>Disease Name</th>
                        <th>Semantics</th>
                        <th>Colors</th>
                    </tr>
                    {% for disease in diseases %}
                    <tr>
                        <td>{{ disease.id }}</td>
                        <td>{{ disease.plant_name }}</td>
                        <td>{{ disease.disease_name }}</td>
                        <td>{{ disease.semantics if disease.semantics else 'N/A' }}</td>
                        <td>{{ disease.colors if disease.colors else 'N/A' }}</td>
                    </tr>
                    {% endfor %}
                </table>

                <h2>Guide for Semantics</h2>
                  <div class="guide-section">
                      <h3>Semantics Definitions</h3>
                      <div class="patterns">
                          <div class="pattern">
                              <strong>Flecks:</strong> Tiny scattered specks on the leaf surface.
                              <img src="{{ url_for('static', filename='images/flecks.JPG') }}" alt="Flecks" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Spots:</strong> Circular or irregularly shaped discolorations.
                              <img src="{{ url_for('static', filename='images/spots.JPG') }}" alt="Spots" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Stripes:</strong> Linear streaks of a different color.
                              <img src="{{ url_for('static', filename='images/stripes.JPG') }}" alt="Stripes" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Powdery:</strong> Dust-like appearance, usually white or gray.
                              <img src="{{ url_for('static', filename='images/powdery.JPG') }}" alt="Powdery" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Velvety:</strong> Soft, thick, and plush texture on the surface.
                              <img src="{{ url_for('static', filename='images/velvety.JPG') }}" alt="Velvety" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Mosaic:</strong> Patchy mix of normal and discolored areas.
                              <img src="{{ url_for('static', filename='images/mosaic.JPG') }}" alt="Mosaic" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Curls:</strong> Twisting or rolling of leaf edges or entire leaf.
                              <img src="{{ url_for('static', filename='images/curls.JPG') }}" alt="Curls" width="75">
                          </div>
                      </div>

                      <h3>Color Spectrums</h3>
                      <div class="color-spectrum">
                          <div class="color-block">
                              <strong>Red:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/red.png') }}" alt="Red" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Brown (includes Orange):</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/brown.png') }}" alt="Brown" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Yellow:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/yellow.png') }}" alt="Yellow" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Pink (includes Purple):</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/pink.png') }}" alt="Pink" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Gray:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/gray.png') }}" alt="Gray" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Black:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/black.png') }}" alt="Black" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>White:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/white.png') }}" alt="White" width="175">
                              </div>
                          </div>

                      </div>
                  </div>

            </div>
        </div>
    </body>
    </html>

    '''
    return render_template_string(html_template, diseases=diseases)

@app.route('/add', methods=['GET'])
def add():
    """Renders the Add New Disease page with a text input for a disease description."""
    add_template = '''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Add New Disease</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 0;
                background: #f7f7f7;
                display: flex;
                justify-content: center;
                align-items: center;
                min-height: 100vh;
            }
            .container {
                width: 80%;
                max-width: 900px;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 4px 10px rgba(0,0,0,0.1);
                text-align: center;
            }
            form {
                display: flex;
                flex-direction: column;
                align-items: center;
                width: 100%;
            }
            label {
                width: 100%;
                text-align: left;
                margin-top: 10px;
                font-weight: bold;
            }
            textarea {
                padding: 8px;
                width: 90%;
                height: 100px;
                border-radius: 5px;
                border: 1px solid #ccc;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
            }
            input[type="submit"] {
                margin-top: 20px;
                padding: 12px 24px;
                background-color: #4CAF50;
                border: none;
                color: white;
                border-radius: 25px;
                cursor: pointer;
                transition: background 0.3s;
            }
            input[type="submit"]:hover {
                background-color: #45a049;
            }
            .button-section {
                margin-top: 20px;
            }
            .button-section a {
                text-decoration: none;
                color: #4CAF50;
                font-weight: bold;
            }
            .guide-section {
                margin-top: 40px;
                text-align: left;
            }
            .patterns {
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(170px, 1fr));
                gap: 15px;
            }
            .pattern {
                background: #f1f1f1;
                padding: 10px;
                border-radius: 8px;
            }
            .pattern img {
                margin-top: 5px;
                width: 100%;
                border-radius: 5px;
                object-fit: cover;
            }
            .color-spectrum {
                display: flex;
                flex-wrap: wrap;
                gap: 30px;
                margin-top: 20px;
            }
            .color-block {
                flex: 1;
                min-width: 200px;
            }
            .color-block div {
                margin-top: 5px;
            }
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Add New Disease</h1>
            <form method="post" action="{{ url_for('entities') }}">
                <label for="raw_text">Enter disease description:</label>
                <textarea name="raw_text" id="raw_text" required></textarea>
                <div class="button-section">
                    <input type="submit" value="Submit">
                </div>
            </form>

            <h2>Guide for Semantics</h2>
                  <div class="guide-section">
                      <h3>Semantics Definitions</h3>
                      <div class="patterns">
                          <div class="pattern">
                              <strong>Flecks:</strong> Tiny scattered specks on the leaf surface.
                              <img src="{{ url_for('static', filename='images/flecks.JPG') }}" alt="Flecks" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Spots:</strong> Circular or irregularly shaped discolorations.
                              <img src="{{ url_for('static', filename='images/spots.JPG') }}" alt="Spots" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Stripes:</strong> Linear streaks of a different color.
                              <img src="{{ url_for('static', filename='images/stripes.JPG') }}" alt="Stripes" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Powdery:</strong> Dust-like appearance, usually white or gray.
                              <img src="{{ url_for('static', filename='images/powdery.JPG') }}" alt="Powdery" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Velvety:</strong> Soft, thick, and plush texture on the surface.
                              <img src="{{ url_for('static', filename='images/velvety.JPG') }}" alt="Velvety" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Mosaic:</strong> Patchy mix of normal and discolored areas.
                              <img src="{{ url_for('static', filename='images/mosaic.JPG') }}" alt="Mosaic" width="75">
                          </div>
                          <div class="pattern">
                              <strong>Curls:</strong> Twisting or rolling of leaf edges or entire leaf.
                              <img src="{{ url_for('static', filename='images/curls.JPG') }}" alt="Curls" width="75">
                          </div>
                      </div>

                      <h3>Color Spectrums</h3>
                      <div class="color-spectrum">
                          <div class="color-block">
                              <strong>Red:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/red.png') }}" alt="Red" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Brown (includes Orange):</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/brown.png') }}" alt="Brown" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Yellow:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/yellow.png') }}" alt="Yellow" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Pink (includes Purple):</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/pink.png') }}" alt="Pink" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Gray:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/gray.png') }}" alt="Gray" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>Black:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/black.png') }}" alt="Black" width="175">
                              </div>
                          </div>
                          <div class="color-block">
                              <strong>White:</strong><br>
                              <div style="display:flex;">
                                  <img src="{{ url_for('static', filename='images/white.png') }}" alt="White" width="175">
                              </div>
                          </div>

                      </div>
                  </div>

            <div class="button-section">
                <a href="{{ url_for('index') }}">Back to Home</a>
            </div>
        </div>
    </body>
    </html>

    '''
    return render_template_string(add_template)

@app.route('/entities', methods=['POST'])
def entities():
    """Analyzes the submitted text and displays the identified entities for confirmation."""
    raw_text = request.form.get("raw_text")
    print("Raw Text:", raw_text)
    extracted = extract_entities(raw_text)
    return render_template_string(entities_template,
                                  plant_name=extracted["plant_name"],
                                  disease_name=extracted["disease_name"],
                                  semantics=extracted["semantics"],
                                  colors=extracted["colors"],
                                  error=None)


@app.route('/confirm', methods=['POST'])
def confirm():
    """
    Inserts the new disease record along with its semantics and colors.
    On success, redirects to the home page.
    On error, re-renders the entities page with an error message.
    """
    plant_name = request.form.get("plant_name")
    disease_name = request.form.get("disease_name")
    semantics = request.form.get("semantics")
    colors = request.form.get("colors")
    color_list = [s.strip() for s in colors.split(",") if s.strip()]

    # If the only color is "black", also add "gray"
    if len(color_list) == 1 and color_list[0].lower() == "black":
        color_list.append("gray")

    try:
        conn = sqlite3.connect(DATABASE)
        c = conn.cursor()

        # Check if the disease for the given plant already exists
        c.execute("SELECT id FROM diseases WHERE plant_name=? AND disease_name=?;",
                  (plant_name, disease_name))
        if c.fetchone():
            conn.close()
            error_message = "Disease for this plant already exists."
            return render_template_string(entities_template,
                                          plant_name=plant_name,
                                          disease_name=disease_name,
                                          semantics=semantics,
                                          colors=colors,
                                          error=error_message)
        conn.close()

        # Call helper function to insert the disease and related records
        add_disease_entry(DATABASE, plant_name, disease_name, semantics, color_list)

        return redirect(url_for('index'))

    except Exception as e:
        error_message = f"Error adding disease: {e}"
        return render_template_string(entities_template,
                                      plant_name=plant_name,
                                      disease_name=disease_name,
                                      semantics=semantics,
                                      colors=colors,
                                      error=error_message)


@app.route('/classify', methods=['GET'])
def classify_page():
    # Get distinct plant names for the dropdown from your database.
    conn = sqlite3.connect(DATABASE)
    conn.row_factory = sqlite3.Row
    c = conn.cursor()
    c.execute("SELECT DISTINCT plant_name FROM diseases;")
    plants = c.fetchall()
    conn.close()


    html_template = '''
    <!DOCTYPE html>
    <html>
    <head>
        <title>Classify a Leaf</title>
        <style>
            body {
                font-family: Arial, sans-serif;
                margin: 0;
                padding: 0;
                background: #f7f7f7;
                display: flex;
                justify-content: center;
                align-items: center;
                min-height: 100vh;
            }
            .container {
                width: 50%;
                max-width: 600px;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 4px 10px rgba(0,0,0,0.1);
                text-align: center;
            }
            form {
                display: flex;
                flex-direction: column;
                align-items: center;
            }
            label {
                width: 100%;
                text-align: left;
                margin-top: 10px;
                font-weight: bold;
            }
            select, input[type="file"] {
                padding: 8px;
                width: 90%;
                border-radius: 5px;
                border: 1px solid #ccc;
                box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.1);
            }
            input[type="submit"] {
                margin-top: 20px;
                padding: 12px 24px;
                background-color: #4CAF50;
                border: none;
                color: white;
                border-radius: 25px;
                cursor: pointer;
                transition: background 0.3s;
            }
            input[type="submit"]:hover {
                background-color: #45a049;
            }
            .result {
                margin-top: 20px;
                font-size: 20px;
                font-weight: bold;
                color: #333;
            }
            .uploaded-image-container {
                margin-top: 20px;
                text-align: center;
            }
            .uploaded-image {
                max-width: 100%;
                height: auto;
                border-radius: 10px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.1);
                display: none;
            }
            .button-section {
                margin-top: 20px;
            }
            .button-section a {
                text-decoration: none;
                color: #4CAF50;
                font-weight: bold;
            }
        </style>
        <script>
            function previewImage(event) {
                const image = document.getElementById('preview');
                image.src = URL.createObjectURL(event.target.files[0]);
                image.style.display = 'block';
            }

            async function classifyImage(event) {
                event.preventDefault();
                const formData = new FormData(document.getElementById('classify-form'));

                const response = await fetch('/api/classify', {
                    method: 'POST',
                    body: formData
                });

                const result = await response.json();
                document.getElementById('result').innerHTML = `
                    <p>Crop Selected: ${result.crop}</p>
                    <p>Semantics Identified: ${result.label}</p>
                    <p>Colors Identified: ${result.colors}</p>
                    <p>Disease Predicted: ${result.disease}</p>
                `;
            }
        </script>
    </head>

    <body>
        <div class="container">
            <h1>Classify a Leaf</h1>
            <form id="classify-form" enctype="multipart/form-data" onsubmit="classifyImage(event)">
                <label for="crop">Name of the Crop:</label>
                <select id="crop" name="crop" required>
                    <option value="">Select Crop</option>
                    {% for row in plants %}
                        <option value="{{ row.plant_name }}">{{ row.plant_name }}</option>
                    {% endfor %}
                </select>
                <label for="image">Attach Image:</label>
                <input type="file" id="image" name="image" accept="image/*" required onchange="previewImage(event)">
                <div class="uploaded-image-container">
                    <img id="preview" class="uploaded-image" alt="Preview Image">
                </div>
                <div class="button-section">
                    <input type="submit" value="Classify">
                </div>
            </form>
            <div id="result" class="result"></div>
            <div class="button-section">
                <a href="/">Back to Home</a>
            </div>
        </div>
    </body>
    </html>
    '''
    return render_template_string(html_template, plants=plants)

@app.route('/api/classify', methods=['POST'])
def api_classify():
    start_time_us = time.time_ns() // 1000
    crop_selected = request.form.get("crop")
    image_file = request.files.get("image")
    predicted_label_text = None
    disease_predicted = None
    colors_found = None

    if image_file and crop_selected:
        temp_path = "temp_image.jpg"
        image_file.save(temp_path)

        print("Starting classification for crop:", crop_selected)
        print("Using image:", temp_path)

        # Load and preprocess the image
        try:
            image = Image.open(temp_path).convert("RGB")
        except Exception as e:
            print("Error opening image:", e)
            predicted_label_text = "Error opening image"
            disease_predicted = "Error"
            colors_found = "Error"
            os.remove(temp_path)
            return jsonify({
                "crop": crop_selected,
                "label": predicted_label_text,
                "disease": disease_predicted,
                "colors": colors_found
            })

        image = image.resize((256, 256))
        image_array = np.array(image)
        image_norm = image_array / 255.0
        input_image = np.expand_dims(image_norm, axis=0)

        print("Running classification model...")
        class_pred = classification_model.predict(input_image)
        predictions = class_pred[0]
        print("Prediction:", predictions)

        # Mapping integer prediction to semantic label
        labels_mapping = {
            0: "curls",
            1: "flecks",
            2: "mosaic",
            3: "powdery",
            4: "spots",
            5: "stripes",
            6: "velvety"
        }

        # Run segmentation for color extraction
        top_candidate = np.argmax(predictions)
        color_counts = {}
        colors_found = "N/A"
        print("Running segmentation for color extraction...")
        seg_pred = semantics_model.predict(input_image)
        seg_mask = seg_pred[0, ..., 0]
        threshold = 0.5
        seg_mask_binary = (seg_mask > threshold).astype(np.float32)
        segmented_image = image_norm.copy()
        segmented_image[seg_mask_binary == 1] = 0
        segmented_image_uint8 = (segmented_image * 255).astype(np.uint8)
        hsv_image = cv2.cvtColor(segmented_image_uint8, cv2.COLOR_RGB2HSV)
        masked_hsv_pixels = hsv_image[seg_mask_binary == 0]

        # Define HSV color ranges


        color_ranges = {
              'red':    (np.array([0, 100, 100]),   np.array([5, 250, 250])),
              'brown':  (np.array([6, 80, 80]),  np.array([20, 250, 250])),
              'yellow': (np.array([25, 80, 80]), np.array([30, 250, 250])),
              'pink':   (np.array([130, 60, 60]),np.array([160, 255, 255])),
              'gray':   (np.array([40, 30, 30]),    np.array([55, 120, 120])),
              'black':  (np.array([0, 0, 0]),      np.array([179, 255, 60])),
              'white':  (np.array([0, 0, 210]),    np.array([179, 30, 255]))
          }

        # Count each color in the masked region
        for color, (lower, upper) in color_ranges.items():
            condition = np.all((masked_hsv_pixels >= lower) & (masked_hsv_pixels <= upper), axis=1)
            count = int(np.sum(condition))
            color_counts[color] = count

        print("Color counts:", color_counts)
        sorted_colors = sorted(color_counts.items(), key=lambda x: x[1], reverse=True)
        color_list = [color for color, count in sorted_colors if count > 0]
        colors_found = ", ".join(color_list[:3])
        #print("Color counts (sorted):", sorted_colors)
        print("Extracted colors (sorted by count):", colors_found)

        # Dynamic Query: Find a Matching Disease Record in the database
        sorted_indices = np.argsort(predictions)[::-1]
        print("Sorted prediction indices:", sorted_indices)
        record_found = None

        record_found = get_best_weighted_iou_match(
        crop_selected,
        labels_mapping.get(sorted_indices[0]),
        colors_list=colors_found
        )


        if record_found:
            disease_predicted = record_found[2]
            print("Matching disease record found:")
            print("  Plant Name:", record_found[1])
            print("  Semantics:", record_found[3])
            print("  Disease Predicted:", record_found[2])
            print("  Record Colors:", record_found[4])
        else:
            disease_predicted = "No matching disease record found"
            print("No matching disease record.")
        """
        print("Final Result:")
        print("  Crop Selected:", crop_selected)
        print("  Label:", record_found[3])
        print("  Disease Predicted:", record_found[2])
        print("  Colors (sorted by count):", record_found[4])
        """
        os.remove(temp_path)
    else:
        predicted_label_text = "No image or crop selected"
        disease_predicted = "N/A"
        colors_found = "N/A"

    result = {
        "crop": crop_selected,
        "label": labels_mapping.get(sorted_indices[0]),
        "disease": disease_predicted,
        "colors": colors_found
    }
    end_time_us = time.time_ns() // 1000
    time_elapsed_us = end_time_us - start_time_us
    print(f"Time elapsed (microseconds): {time_elapsed_us}")
    return jsonify(result)


def run_flask():
    app.run(port=5023)

# Run Flask in a separate thread
thread = threading.Thread(target=run_flask)
thread.start()

/usr/local/lib/python3.11/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_pipeline' (0.0.0) was trained with spaCy v3.7.5 and may not be 100% compatible with the current version (3.8.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


 * Serving Flask app '__main__'
 * Debug mode: off


In [ ]:
!./ngrok http 5023

INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:29] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:29] "GET /static/images/flecks.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/spots.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/velvety.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/black.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/powdery.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/curls.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/pink.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/stripes.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:30] "GET /static/images/brown.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:

Raw Text: apple plants with blackscab have black spots
Identified entity, Label: PLANT, Entity: apple
Identified entity, Label: COLOR, Entity: black
Identified entity, Label: SEMANTIC, Entity: spots


INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:51] "POST /confirm HTTP/1.1" 302 -


✅ Disease 'blackscab' added successfully.


INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/flecks.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/velvety.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/mosaic.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/curls.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/pink.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/red.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/spots.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/white.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "GET /static/images/black.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:38:52] "

...

✅ Disease 'cedarrust' added successfully.


INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/spots.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/flecks.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/stripes.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/velvety.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/curls.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/red.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/yellow.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/brown.png HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:40] "GET /static/images/powdery.JPG HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:39:

Starting classification for crop: apple
Using image: temp_image.jpg
Running classification model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Prediction: [3.9686707e-09 2.3954337e-04 7.0836731e-12 2.4889095e-08 9.9970692e-01
 5.3522192e-05 1.6960348e-10]
Running segmentation for color extraction...
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


INFO:werkzeug:127.0.0.1 - - [25/May/2025 05:40:00] "POST /api/classify HTTP/1.1" 200 -


Color counts: {'red': 0, 'brown': 131, 'yellow': 488, 'pink': 0, 'gray': 0, 'black': 291, 'white': 0}
Extracted colors (sorted by count): yellow, black, brown
Sorted prediction indices: [4 1 5 3 0 6 2]
Plant name:  apple
Semantics:  spots
colors list: ['yellow', 'black', 'brown']
weighted dictionary : {'yellow': 1.0, 'black': 0.8, 'brown': 0.6}
normalized colors : ['yellow', 'black', 'brown']
Matching disease record found:
  Plant Name: apple
  Semantics: spots
  Disease Predicted: cedarrust
  Record Colors: brown, yellow
Time elapsed (microseconds): 4597417
